# Food.com Data Pipeline

This notebook is a thin wrapper around `foodcom_data_pipeline.py`, which is the source of truth for the Data workflow.

Running the notebook will:
- clean recipes and interactions
- align interactions to cleaned recipe ids
- build temporal train/test splits
- create support-filtered collaborative-filtering splits
- regenerate the summary JSON files in `Final/Data/Pure_Data`


In [ ]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)


In [ ]:
cwd = Path.cwd().resolve()
code_dir_candidates = [cwd, cwd / "Final" / "Data" / "Data_Code"]

for candidate in code_dir_candidates:
    if (candidate / "foodcom_data_pipeline.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not locate foodcom_data_pipeline.py")

from foodcom_data_pipeline import build_config, run_pipeline


In [ ]:
MIN_USER_RATINGS = 5
MIN_RECIPE_RATINGS = 5
FORCE_DOWNLOAD = False

config = build_config(
    start=Path.cwd(),
    min_user_ratings=MIN_USER_RATINGS,
    min_recipe_ratings=MIN_RECIPE_RATINGS,
    force_download=FORCE_DOWNLOAD,
)

config


In [ ]:
results = run_pipeline(config)
summary = results["summary"]
temporal_split_summary = results["temporal_split_summary"]


In [ ]:
pd.DataFrame(
    [
        {"section": "preprocessing", **summary},
        {"section": "temporal_split", **temporal_split_summary},
    ]
).T


In [ ]:
sorted(path.name for path in config.pure_dir.glob("*"))


In [ ]:
recipe_columns_to_preview = [
    "id",
    "name",
    "clean_rating_count",
    "filtered_rating_count",
    "history_bucket",
    "eligible_for_cf",
]

results["recipe_model_table"][recipe_columns_to_preview].head()


In [ ]:
user_columns_to_preview = [
    "user_id",
    "clean_user_rating_count",
    "filtered_user_rating_count",
    "history_bucket",
    "eligible_for_cf",
]

results["user_statistics"][user_columns_to_preview].head()
